---
title: Word Game Hacks/Changes
description: Learn to code Wordle in Javascript and implement core concepts in software with comments
comments: false
layout: post
permalink: /Word_Game/lesson
---

# Hack 1: Progress Bar  
I added a progress bar below the canvas that fills up as the player types more characters.  
It visually shows typing progress in real-time and resets when the game restarts.  


In [ ]:
// === Hack #1: Progress Bar ===
const progressBar = document.createElement('div');
progressBar.style.width = '0%';
progressBar.style.height = '20px';
progressBar.style.backgroundColor = '#28a745';
progressBar.style.marginTop = '10px';
progressBar.style.borderRadius = '5px';

const progressContainer = document.createElement('div');
progressContainer.style.width = '800px';
progressContainer.style.height = '20px';
progressContainer.style.backgroundColor = '#ddd';
progressContainer.style.margin = '10px auto';
progressContainer.style.borderRadius = '5px';
progressContainer.appendChild(progressBar);

document.body.insertBefore(progressContainer, wordCanvas.nextSibling);

function updateProgress(prompt, input) {
    const progressPercent = (input.length / prompt.length) * 100;
    progressBar.style.width = progressPercent + '%';
}


# Hack 2: Current Character Highlighter  
I added a small rectangle that highlights the next character the user must type.  
It moves dynamically as the player types, helping focus attention and improving accuracy.  


In [ ]:
// === Hack #2: Current Character Highlighter ===
function drawUserTextWithHighlight(prompt, input) {
    wordCtx.clearRect(0, 0, wordCanvas.width, wordCanvas.height);
    wordCtx.font = '24px Arial';
    wordCtx.textAlign = 'left';

    const maxWidth = wordCanvas.width - 20;
    const lineHeight = 30;
    const lines = wrapText(prompt, maxWidth);
    const startY = (wordCanvas.height - lines.length * lineHeight) / 2;

    let charIndex = 0;
    lines.forEach((line, lineIndex) => {
        const lineY = startY + lineIndex * lineHeight;
        const lineX = (wordCanvas.width - wordCtx.measureText(line).width) / 2;
        wordCtx.fillStyle = '#dededeff';
        wordCtx.fillText(line, lineX, lineY);

        // highlight the next character box
        if (charIndex <= input.length && input.length < prompt.length) {
            const nextChar = prompt[input.length];
            const before = line.slice(0, input.length - charIndex);
            const highlightX = lineX + wordCtx.measureText(before).width;

            if (prompt[input.length] && line.includes(nextChar)) {
                wordCtx.strokeStyle = 'orange';
                wordCtx.lineWidth = 2;
                wordCtx.strokeRect(
                    highlightX - 2,
                    lineY - 20,
                    wordCtx.measureText(nextChar).width + 4,
                    24
                );
            }
        }

        // draw user input
        let currentX = lineX;
        for (let i = 0; i < line.length && charIndex < input.length; i++, charIndex++) {
            const char = input[charIndex];
            const promptChar = prompt[charIndex];
            const color = char === promptChar ? 'green' : 'red';
            wordCtx.fillStyle = color;
            wordCtx.fillText(char, currentX, lineY);
            currentX += wordCtx.measureText(promptChar).width;
        }
        charIndex += Math.max(0, line.length - (input.length - charIndex));
    });
}


# Hack 3: Dark Mode Toggle  
I added a "Dark Mode" button that changes the background and text colors of the game.  
This makes the game more customizable and user-friendly, especially for long sessions.  


In [ ]:
// === Hack #3: Dark Mode Toggle ===
const darkModeButton = document.createElement('button');
darkModeButton.textContent = 'Toggle Dark Mode';
darkModeButton.style.display = 'block';
darkModeButton.style.margin = '20px auto';
darkModeButton.style.padding = '10px 20px';
darkModeButton.style.backgroundColor = '#444';
darkModeButton.style.color = '#fff';
darkModeButton.style.border = 'none';
darkModeButton.style.borderRadius = '5px';
document.body.insertBefore(darkModeButton, progressContainer);

let darkMode = false;
darkModeButton.addEventListener('click', () => {
    darkMode = !darkMode;
    document.body.style.backgroundColor = darkMode ? '#121212' : '#ffffff';
    document.body.style.color = darkMode ? '#e0e0e0' : '#000000';
    wordCanvas.style.borderColor = darkMode ? '#ffffff' : '#000000';
});


Integration Notes  
I updated `startGame()` and `document.onkeydown` to:  
- call `updateProgress(selectedString, userInput)` each time input changes.  
- use `drawUserTextWithHighlight(selectedString, userInput)` instead of the old `drawUserText()`.  

This ensures the progress bar and character highlighter update in real-time.  


In [ ]:
// === Integration updates ===
function startGame() {
    if (currentString === "") {
        alert("Please select a string length from the options menu.");
        return;
    }

    let stringArray;
    if (currentString === "short_strings") {
        stringArray = short_strings;
    } else if (currentString === "medium_strings") {
        stringArray = medium_strings;
    } else if (currentString === "long_strings") {
        stringArray = long_strings;
    }

    const randomIndex = Math.floor(Math.random() * stringArray.length);
    const selectedString = stringArray[randomIndex];
    userInput = "";
    mistakes = 0;
    finished = false;
    startTime = Date.now();
    drawText(selectedString);
    document.querySelector('.wpm').textContent = '0';
    document.querySelector('.accuracy').textContent = '100%';
    progressBar.style.width = '0%'; // reset progress bar

    document.onkeydown = function (e) {
        if (finished) return;

        if (e.key.length === 1 && userInput.length < selectedString.length) {
            const nextChar = selectedString[userInput.length];
            if (e.key !== nextChar) {
                mistakes++;
            }
            userInput += e.key;
        } else if (e.key === 'Backspace' && userInput.length > 0) {
            userInput = userInput.slice(0, -1);
        }

        drawUserTextWithHighlight(selectedString, userInput);
        updateStats(selectedString, userInput, startTime);
        updateProgress(selectedString, userInput);

        if (userInput === selectedString) {
            finishGame(selectedString, userInput, startTime);
        }
    };
}
